# Custom Agent Roles and System Messages

A multi-agent team is only as good as each agent's **role contract**. In AgentChat, two layers define behavior:

1. **`system_message`** — instructions the agent sees every turn (planner vs executor vs reviewer).
2. **`name` + `description`** — metadata the **team selector** uses to route (**09**).

This notebook designs **Notes API specialist personas**, previews **`HandoffMessage`** and the **`Swarm`** pattern, and catalogs **anti-patterns** that break routing or tool use.

**Prerequisites:** **03. AssistantAgent**, **09. GroupChatManager and Speaker Selection**, **06. Tools and Function Registration**.

**Cross-references:** **11. Sequential and Hierarchical Workflows**, **LangGraph/11** (supervisor workers), **13. Termination Conditions**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("pip install autogen-agentchat autogen-ext[openai]")

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.base import Handoff
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import HandoffMessage, TextMessage
from autogen_agentchat.teams import SelectorGroupChat, Swarm
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import UserMessage

In [ ]:
DOC_CHUNKS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]


def search_docs(query: str) -> str:
    """Keyword search over Notes API documentation chunks."""
    tokens = set(query.lower().split())
    hits = []
    for chunk in DOC_CHUNKS:
        if tokens & set(chunk["text"].lower().split()):
            hits.append(f"[{chunk['id']}] {chunk['text']}")
    return "\n".join(hits) if hits else "No matching chunks."


def describe_endpoint(method: str, path: str) -> str:
    """Describe a Notes API HTTP endpoint (teaching stub)."""
    catalog = {
        ("GET", "/notes"): "List all notes for the authenticated user.",
        ("POST", "/notes"): "Create a note; body requires title and content.",
        ("PUT", "/notes/{id}"): "Update an existing note by id.",
        ("DELETE", "/notes/{id}"): "Delete a note by id.",
    }
    return catalog.get((method.upper(), path), f"Unknown endpoint {method} {path}")

In [ ]:
def make_model_client():
    return OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

model_client = make_model_client()

---

## 1. Role Triangle — Planner, Executor, Reviewer

| Role | Responsibility | Tools? | Speaks when |
|------|----------------|--------|-------------|
| **Planner** | Decompose task, assign work, summarize | Usually no | Task start / checkpoint / end |
| **Executor** | Run tools, return evidence | Yes | After assignment |
| **Reviewer** | Validate quality, request fixes | Optional | Before TERMINATE |

```text
User → Planner → Executor (tools) → Reviewer → Planner → TERMINATE
```

In `SelectorGroupChat`, encode this flow in **descriptions** and **system_message** boundaries.

---

## 2. Planner System Message Template

Planners should **delegate**, not **execute**:

In [ ]:
PLANNER_SYSTEM = """
You are the Notes API planning agent (notes_planner).

Team:
- docs_executor: searches documentation with search_docs
- api_executor: describes HTTP routes with describe_endpoint
- review_agent: validates answers before finish

Rules:
1. On a new task, output numbered assignments: <agent_name>: <subtask>
2. Never call tools yourself.
3. After review_agent approves, summarize with citations and end with TERMINATE.
4. Keep assignments atomic — one tool purpose per executor turn.
""".strip()

print(PLANNER_SYSTEM[:120], "...")

---

## 3. Executor System Message Template

Executors should be **narrow** and **tool-grounded**:

In [ ]:
DOCS_EXECUTOR_SYSTEM = """
You are docs_executor for the FastAPI Notes API.
Use search_docs for every factual claim about the codebase stack.
Cite chunk ids like [c2]. Do not invent endpoints or commands.
If the task needs HTTP paths, say so and stop — do not guess routes.
""".strip()

API_EXECUTOR_SYSTEM = """
You are api_executor for the Notes API.
Use describe_endpoint(method, path) for route behavior.
Do not search docs unless the planner explicitly asks.
""".strip()

---

## 4. Reviewer System Message Template

Reviewers catch hallucinations before termination:

In [ ]:
REVIEWER_SYSTEM = """
You are review_agent. You do not call tools.
Check that:
- doc claims include [c#] citations
- endpoint answers name method + path
- nothing contradicts the Notes API teaching corpus
Reply APPROVED if sufficient, else list concrete fixes needed.
""".strip()

---

## 5. `name` and `description` for Team Routing

**`description`** is shown to the selector LLM (**09**). Keep it **action-oriented**:

| Field | Planner example |
|-------|-----------------|
| `name` | `notes_planner` |
| `description` | `Breaks down Notes API tasks and assigns executors. Should speak first and last.` |

Poor description: `"An AI agent"` — the selector cannot route.

In [ ]:
def make_notes_role_agents():
    planner = AssistantAgent(
        "notes_planner",
        description="Breaks down Notes API tasks and assigns executors. Speaks first and last.",
        model_client=model_client,
        system_message=PLANNER_SYSTEM,
    )
    docs = AssistantAgent(
        "docs_executor",
        description="Searches Notes API documentation chunks using search_docs.",
        model_client=model_client,
        tools=[search_docs],
        system_message=DOCS_EXECUTOR_SYSTEM,
    )
    api = AssistantAgent(
        "api_executor",
        description="Describes Notes API HTTP endpoints using describe_endpoint.",
        model_client=model_client,
        tools=[describe_endpoint],
        system_message=API_EXECUTOR_SYSTEM,
    )
    reviewer = AssistantAgent(
        "review_agent",
        description="Reviews executor output for citations and accuracy before finish.",
        model_client=model_client,
        system_message=REVIEWER_SYSTEM,
    )
    return planner, docs, api, reviewer


agents = make_notes_role_agents()
print("roles:", [a.name for a in agents])

---

## 6. Assemble Role-Based `SelectorGroupChat`

In [ ]:
termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=24)

role_team = SelectorGroupChat(
    list(agents),
    model_client=model_client,
    termination_condition=termination,
)

ROLE_TASK = (
    "Explain JWT authentication for the Notes API and describe the GET /notes endpoint."
)

# await Console(role_team.run_stream(task=ROLE_TASK))
print("role_team ready")

---

## 7. `HandoffMessage` — Control Transfer Preview

`HandoffMessage` signals **which agent speaks next** in **`Swarm`** teams. Fields: `source`, `target`, `content`:

In [ ]:
handoff = HandoffMessage(
    source="notes_planner",
    target="docs_executor",
    content="Please search documentation for JWT bearer authentication.",
)

print("type:", handoff.type)
print("target:", handoff.target)
print("content:", handoff.content)
print("json:", handoff.model_dump())

In a running team, the **current speaker** emits a `HandoffMessage` (often via the `handoffs=` config on `AssistantAgent`) instead of relying on a central selector.

---

## 8. `Handoff` Config on `AssistantAgent` (Swarm Prep)

`Handoff` objects declare valid transfer targets and customize handoff text:

In [ ]:
handoffs = [
    Handoff(target="docs_executor", message="Assign documentation search."),
    Handoff(target="api_executor", message="Assign endpoint description."),
    Handoff(target="review_agent", message="Request review of collected answers."),
]

planner_with_handoffs = AssistantAgent(
    "notes_planner",
    description="Delegates via handoffs to executors and reviewer.",
    model_client=model_client,
    system_message=PLANNER_SYSTEM + "\nUse handoffs to assign work.",
    handoffs=handoffs,
)
print("handoff targets:", [h.target for h in handoffs])

---

## 9. `Swarm` Pattern — Decentralized Routing

**`Swarm`** (OpenAI Swarm-style) picks the next speaker from the latest **`HandoffMessage`**, not a selector LLM:

| Team | Who decides next speaker? |
|------|---------------------------|
| `RoundRobinGroupChat` | Turn index |
| `SelectorGroupChat` | Selector LLM / `selector_func` |
| `Swarm` | Latest `HandoffMessage.target` |

First participant in the list is the **initial speaker**. See **11** for when to prefer centralized vs decentralized routing.

In [ ]:
planner, docs, api, reviewer = make_notes_role_agents()

# Re-create planner with handoffs for Swarm demo
planner_swarm = AssistantAgent(
    "notes_planner",
    description="Delegates via handoffs.",
    model_client=model_client,
    system_message=PLANNER_SYSTEM,
    handoffs=[
        Handoff(target="docs_executor"),
        Handoff(target="api_executor"),
        Handoff(target="review_agent"),
    ],
)

swarm_team = Swarm(
    [planner_swarm, docs, api, reviewer],
    termination_condition=termination,
)
print("Swarm team:", type(swarm_team).__name__)

---

## 10. Persona Templates — Notes API Specialists

Copy-ready personas for your own projects:

In [ ]:
PERSONA_TEMPLATES = {
    "planner": {
        "name": "notes_planner",
        "description": "Coordinates Notes API tasks; speaks first and last.",
        "system_message": PLANNER_SYSTEM,
    },
    "docs": {
        "name": "docs_executor",
        "description": "Documentation retrieval via search_docs.",
        "system_message": DOCS_EXECUTOR_SYSTEM,
    },
    "api": {
        "name": "api_executor",
        "description": "HTTP route reference via describe_endpoint.",
        "system_message": API_EXECUTOR_SYSTEM,
    },
    "reviewer": {
        "name": "review_agent",
        "description": "Validates citations and completeness.",
        "system_message": REVIEWER_SYSTEM,
    },
}

print(json.dumps({k: v["name"] for k, v in PERSONA_TEMPLATES.items()}, indent=2))

---

## 11. Build Agent from Persona Template

In [ ]:
def agent_from_persona(key: str, *, tools=None, handoffs=None) -> AssistantAgent:
    p = PERSONA_TEMPLATES[key]
    return AssistantAgent(
        p["name"],
        description=p["description"],
        model_client=model_client,
        system_message=p["system_message"],
        tools=tools or [],
        handoffs=handoffs or [],
    )


demo = agent_from_persona("docs", tools=[search_docs])
print(demo.name, "tools:", len(demo._tools) if hasattr(demo, "_tools") else "configured")

---

## 12. Anti-Patterns in System Prompts

| Anti-pattern | Symptom | Fix |
|--------------|---------|-----|
| Same instructions on every agent | Selector confusion, duplicated work | Split planner vs executor prompts |
| Missing `description` | Wrong speaker in **09** | Add routing-oriented description |
| "You can do everything" | Tool misuse / skipped specialists | Narrow tool + role boundaries |
| TERMINATE in executor prompt | Premature stop | Only planner/reviewer may terminate |
| Huge system message | Token bloat, ignored rules | Short rules + examples |
| Conflicting team roster | Planner names agents not in team | Keep roster in sync with participants |

```text
Bad:  "You are a helpful assistant that can search and code and review."
Good: "You are docs_executor. Only use search_docs. Do not describe HTTP routes."
```

---

## 13. Separation of Concerns Checklist

Before `team.run_stream`:

- [ ] Planner has **no tools** (unless you intend otherwise)
- [ ] Each executor has **only** its tools
- [ ] `description` mentions **when** to select this agent
- [ ] Termination keyword owned by **one** role (**13**)
- [ ] Agent names in planner text **match** `name=` exactly

In [ ]:
def validate_roster(team_agents, planner_text: str):
    names = {a.name for a in team_agents}
    missing = [n for n in names if n not in planner_text]
    return {"team_names": names, "planner_mentions_all": len(missing) == 0}


planner, docs, api, reviewer = make_notes_role_agents()
print(validate_roster([planner, docs, api, reviewer], PLANNER_SYSTEM))

---

## 14. `reflect_on_tool_use` for Natural Language Tool Output

By default, `AssistantAgent` may return raw tool strings. Set `reflect_on_tool_use=True` when executors should **narrate** tool results for other agents:

In [ ]:
docs_reflect = AssistantAgent(
    "docs_executor",
    description="Documentation search with natural language summaries.",
    model_client=model_client,
    tools=[search_docs],
    system_message=DOCS_EXECUTOR_SYSTEM,
    reflect_on_tool_use=True,
)
print("reflect_on_tool_use:", True)

---

## 15. Compare Selector vs Swarm for Same Roles

| Criterion | `SelectorGroupChat` | `Swarm` |
|-----------|--------------------|---------|
| Routing | Central LLM | Agent handoffs |
| Planner must know handoff API | No | Yes (`handoffs=`) |
| Debugging routing | Read selector prompt / func | Read `HandoffMessage` chain |
| Extra LLM calls | Selection each turn | No selector call |

Use **Selector** when roles are fluid; use **Swarm** when agents already know delegation protocol.

---

## 16. `build_notes_specialist_team()` Factory

In [ ]:
def build_notes_specialist_team(pattern: str = "selector"):
    planner, docs, api, reviewer = make_notes_role_agents()
    if pattern == "swarm":
        planner = AssistantAgent(
            "notes_planner",
            description="Delegates via handoffs.",
            model_client=model_client,
            system_message=PLANNER_SYSTEM,
            handoffs=[Handoff(target="docs_executor"), Handoff(target="api_executor"), Handoff(target="review_agent")],
        )
        return Swarm([planner, docs, api, reviewer], termination_condition=termination)
    return SelectorGroupChat([planner, docs, api, reviewer], model_client=model_client, termination_condition=termination)


print("selector:", type(build_notes_specialist_team("selector")).__name__)
print("swarm:", type(build_notes_specialist_team("swarm")).__name__)

---

## 17. Async Demo — Role Team

In [ ]:
async def run_role_team():
    team = build_notes_specialist_team("selector")
    await Console(team.run_stream(task="Document Alembic autogenerate and describe PUT /notes/{id}."))


# await run_role_team()
print("await run_role_team()")

---

## 18. Summary

- **`system_message`** = behavioral contract per role (planner / executor / reviewer).
- **`description`** = routing contract for **09** selector teams.
- **`HandoffMessage` + `Swarm`** = decentralized control transfer (**11** compares workflows).
- Avoid bloated, overlapping prompts — keep roles narrow.

**Next:** **11. Sequential and Hierarchical Workflows** — chain agents with termination, manager delegation, and `MagenticOneGroupChat`.